# ZOD03 — Full 110% checkpoint cascade

Runs the complete 36-rule cascade selected by the 110% support-growth rule, with the recorded fallback. It caches top-1 predictions from each of the 22 distinct checkpoints over the complete competition test set before routing. The rules and fallback were selected using frozen V0 validation results (345/540, 63.89%), so this is a development-set score, not an independent generalization estimate. Internet is disabled; base models are attached as Kaggle model/data sources.

In [ ]:
from pathlib import Path
import gc, json, os, subprocess, sys, time
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
INPUT_ROOT = Path("/kaggle/input")
bundle_manifests = list(INPUT_ROOT.rglob("zod03_manifest.json"))
assert len(bundle_manifests) == 1, f"expected one ZOD03 asset manifest; found {bundle_manifests}"
DATA = bundle_manifests[0].parent
BUNDLE = json.loads((DATA / "zod03_manifest.json").read_text())
from importlib.metadata import version
SKLEARN_VERSION = "1.9.1"
if version("scikit-learn") != SKLEARN_VERSION:
    wheel = next(Path("/kaggle/input").rglob("scikit_learn-1.9.1-*.whl"), None)
    assert wheel is not None, "pinned scikit-learn wheel missing"
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--force-reinstall", str(wheel)], check=True)
import joblib, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "Kaggle GPU unavailable"
WORK = Path("/kaggle/working/zod03_runtime")
SRC = WORK / "src"
SRC.mkdir(parents=True, exist_ok=True)
for name in ("__init__.py", "data_factory.py", "feature_factory.py", "model_factory.py", "inference.py"):
    (SRC / name).symlink_to(DATA / f"source__{name}")
CHECKPOINTS = WORK / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
BASES = {}
for seq, candidate in BUNDLE["candidates"].items():
    dest = CHECKPOINTS / seq
    dest.mkdir(parents=True, exist_ok=True)
    for item in candidate["files"]:
        source = DATA / item["flat_filename"]
        assert source.is_file(), f"checkpoint asset missing: {seq}/{item['flat_filename']}"
        target = dest / item["relative_path"]
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
    candidate["runtime_checkpoint"] = str(dest if candidate["checkpoint_type"] == "directory" else dest / candidate["files"][0]["relative_path"])
    if candidate.get("base_files") and candidate["model"] not in BASES:
        bdir = WORK / "bases" / candidate["model"]
        bdir.mkdir(parents=True, exist_ok=True)
        for item in candidate["base_files"]:
            source = DATA / item["flat_filename"]
            assert source.is_file(), f"base asset missing: {item['flat_filename']}"
            target = bdir / item["relative_path"]
            target.parent.mkdir(parents=True, exist_ok=True)
            target.symlink_to(source)
        BASES[candidate["model"]] = bdir
def resolve_model(marker):
    for p in Path("/kaggle/input").rglob("config.json"):
        if marker.lower() in str(p.parent).lower():
            return p.parent
    raise FileNotFoundError(f"model base not found under /kaggle/input: {marker}")
for model, marker in (("qwen06", "qwen3-0.6b"), ("gemma4_e2b", "gemma-4-e2b"),
                      ("embeddinggemma2", "embeddinggemma-2"),
                      ("ministral3_3b", "ministral-3-3b-base-2512"), ("qwen17", "1.7b-base")):
    if model in {c["model"] for c in BUNDLE["candidates"].values()} and model not in BASES:
        BASES[model] = resolve_model(marker)
llama_manifests = list(INPUT_ROOT.rglob("llama_base_manifest.json"))
assert len(llama_manifests) == 1, f"expected one exact Llama base manifest; found {llama_manifests}"
LLAMA_DATA = llama_manifests[0].parent
if "llama32_3b" in {c["model"] for c in BUNDLE["candidates"].values()}:
    llama_info = json.loads((LLAMA_DATA / "llama_base_manifest.json").read_text())
    assert llama_info["model"] == "meta-llama/Llama-3.2-3B"
    assert llama_info["revision"] == BUNDLE["candidates"][next(s for s,c in BUNDLE["candidates"].items() if c["model"] == "llama32_3b")]["base_model_revision"]
    llama_dir = WORK / "bases" / "llama32_3b"
    llama_dir.mkdir(parents=True, exist_ok=True)
    for item in llama_info["files"]:
        source = LLAMA_DATA / item["flat_filename"]
        target = llama_dir / item["relative_path"]
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
    BASES["llama32_3b"] = llama_dir
try:
    from importlib.metadata import PackageNotFoundError
    from packaging.version import Version
    if Version(version("torchao")) <= Version("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True, capture_output=True)
except (PackageNotFoundError, ImportError):
    pass
sys.path.insert(0, str(WORK))
from src.inference import _predict_neural, predict_catboost_stylometry, predict_svc_bundle
TEST = next((p for p in (Path("/kaggle/input/competitions/name-that-llm/test.csv"), Path("/kaggle/input/name-that-llm/test.csv")) if p.is_file()), None)
assert TEST is not None, "competition test.csv missing"
assert BUNDLE["algorithm"] == "zod03_110_full_cascade"
assert len(BUNDLE["rules"]) == 36 and len(BUNDLE["candidates"]) == 22
print({"approach": BUNDLE["algorithm"], "applied_rules": len(BUNDLE["rules"]), "checkpoints": len(BUNDLE["candidates"]), "fallback": BUNDLE["fallback_sequence_id"]})

In [ ]:
test = pd.read_csv(TEST, dtype={"id": "string"})
assert {"id", "text"}.issubset(test.columns)
assert test.id.notna().all() and not test.id.duplicated().any()
test.text = test.text.fillna("").astype(str)
classes = list(map(str, BUNDLE["classes"]))
texts, n = test.text.tolist(), len(test)
def get_top1(seq):
    c = BUNDLE["candidates"][seq]
    model, cp = c["model"], Path(c["runtime_checkpoint"])
    if model == "svc":
        probs, labels = predict_svc_bundle(joblib.load(cp), texts)
    elif model == "catboost_stylometry":
        probs, labels = predict_catboost_stylometry(cp, texts, classes)
    else:
        base = BASES.get(model)
        if model != "ultrabert380":
            assert base is not None, f"base model not resolved for {model}"
        bs = min(4, max(1, int(c["config"].get("batch_size", 4) or 4)))
        while True:
            try:
                probs, labels = _predict_neural(model, cp, texts, classes, batch_size=bs,
                    max_length=int(c["config"].get("max_length", 384)),
                    base_checkpoint=str(base) if base else None, revision=None,
                    text_only_backbone=bool(c["config"].get("text_only_backbone", False)))
                break
            except RuntimeError as e:
                if "out of memory" not in str(e).lower() or bs <= 1: raise
                bs = max(1, bs // 2); gc.collect(); torch.cuda.empty_cache()
    labels = list(map(str, labels))
    order = [labels.index(label) for label in classes]
    top = np.asarray(classes, dtype=object)[np.asarray(probs)[:, order].argmax(axis=1)]
    del probs; gc.collect(); torch.cuda.empty_cache()
    return top
started = time.time()
# Run each checkpoint once on the entire test set, then cache top-1 labels.
checkpoint_order = list(dict.fromkeys([r["sequence_id"] for r in BUNDLE["rules"]] + [BUNDLE["fallback_sequence_id"]]))
assert set(checkpoint_order) == set(BUNDLE["candidates"])
top1_cache = {seq: get_top1(seq) for seq in checkpoint_order}
assigned, output = np.zeros(n, dtype=bool), np.empty(n, dtype=object)
source = np.empty(n, dtype=object)
trace = []
for rule in BUNDLE["rules"]:
    seq, label = rule["sequence_id"], rule["label"]
    chosen = np.flatnonzero((~assigned) & (top1_cache[seq] == label))
    assigned[chosen], output[chosen], source[chosen] = True, label, seq
    trace.append({"order": rule["order"], "sequence_id": seq, "label": label,
        "precision_v0": rule["precision"], "smoothed_precision_v0": rule["smoothed_precision"],
        "assigned": len(chosen), "remaining": int((~assigned).sum())})
remaining = np.flatnonzero(~assigned)
fallback = BUNDLE["fallback_sequence_id"]
output[remaining], source[remaining] = top1_cache[fallback][remaining], fallback
submission = pd.DataFrame({"id": test.id, "model": output})
assert submission.columns.tolist() == ["id", "model"] and len(submission) == len(test)
assert submission.id.astype(str).tolist() == test.id.astype(str).tolist()
assert not submission.id.duplicated().any() and submission.id.notna().all()
assert submission.model.notna().all() and submission.model.isin(classes).all()
out = Path("/kaggle/working/submission.parquet")
submission.to_parquet(out, index=False)
pd.DataFrame(trace).to_csv("/kaggle/working/zod03_rule_counts.csv", index=False)
pd.DataFrame({"id": test.id, "model": submission.model, "selected_sequence_id": source}).to_parquet("/kaggle/working/zod03_decisions.parquet", index=False)
print({"approach": "zod03 full 110% cascade", "rows": len(submission), "output": str(out),
       "runtime_seconds": round(time.time()-started, 1), "schema": submission.columns.tolist(),
       "cached_checkpoints": len(top1_cache)})
display(pd.DataFrame(trace)); display(submission.head())